# G1 × SIX — Colab v3 (Python 3.13.x) 

Hai ô kiểm thử tách biệt. Ô 1 chỉ tải file script ~7 KB theo Git blob, biên dịch và chạy tiền kiểm (preflight), **không tải Qwen**. Ô 2 chỉ cho phép chạy nếu preflight PASS và cùng runtime, sẽ tải trọng số khoảng 1,5 GB rồi thực hiện đúng một lượt kiểm thử tensor tổng hợp. Không train, không dùng dữ liệu khoa học, không ghi Google Drive.

V1 và v2 STOP phải giữ nguyên. Phiên bản Python `.patch` có thể thay đổi trong nhánh 3.13; v3 chỉ cho phép Python 3.13 ổn định, ghi đủ phiên bản và đường dẫn thực tế. Nếu ô 1 STOP thì **KHÔNG chạy ô 2**. Một-shot marker ngăn chạy lại ô 2 trong cùng phiên. Khi có receipt, gửi về để thẩm định. Không dùng PASS kỹ thuật làm giấy phép khoa học.


In [ ]:
# G1 v3: model-free preflight, exact Git source and AST/compile
import ast, hashlib, importlib.metadata as md, json, pathlib, platform
import py_compile, shutil, subprocess, sys, urllib.request

SCRIPT_COMMIT="8b139510f55f183e4dfea60f438bad635c87c116"
SCRIPT_BLOB="f394293a7c43d8c726d88a1a9b9f63ab49fe3d4a"
SCRIPT_PATH="scripts/research/six_ttm/g1_qwen06_zero_science_smoke_colab_env_20261109_v3.py"
URL=f"https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/{SCRIPT_COMMIT}/{SCRIPT_PATH}"
ROOT=pathlib.Path("/content/ttm_six_g1_v3")
print("G1_V3_NOTEBOOK_MODE=ZERO_SCIENCE_PREFLIGHT")
print("G1_V3_NOTEBOOK_PYTHON="+platform.python_version())
print("G1_V3_NOTEBOOK_PYTHON_EXE="+sys.executable)
for name in ("torch","transformers","huggingface-hub","tokenizers","safetensors"):
    try: version=md.version(name)
    except md.PackageNotFoundError: version="NOT_INSTALLED"
    print("G1_V3_PACKAGE_"+name+"="+version)
if pathlib.Path("/content").is_dir():
    print("G1_V3_DISK_FREE_BYTES="+str(shutil.disk_usage("/content").free))
source=urllib.request.urlopen(URL,timeout=45).read()
blob=hashlib.sha1(b"blob "+str(len(source)).encode("ascii")+b"\0"+source).hexdigest()
print("G1_V3_SCRIPT_BLOB="+blob)
if blob!=SCRIPT_BLOB: raise RuntimeError("STOP_SOURCE_HASH_MISMATCH")
ast.parse(source.decode("utf-8"),filename=SCRIPT_PATH)
if not pathlib.Path("/content").is_dir(): raise RuntimeError("STOP_NOT_COLAB")
ROOT.mkdir(parents=True,exist_ok=True)
path=ROOT/"g1_qwen06_zero_science_smoke_colab_env_20261109_v3.py"
path.write_bytes(source)
py_compile.compile(str(path),doraise=True)
print("G1_V3_STATIC_COMPILE=PASS")
proc=subprocess.run([sys.executable,str(path),"--phase","preflight"],check=False)
print("G1_V3_PREFLIGHT_EXIT="+str(proc.returncode))
receipt=ROOT/"G1_PREFLIGHT_RECEIPT.json"
if receipt.is_file():
    record=json.loads(receipt.read_text(encoding="utf-8"))
    print("G1_V3_PREFLIGHT_RECEIPT="+json.dumps(record,sort_keys=True))
else:
    print("G1_V3_PREFLIGHT_RECEIPT=MISSING")
if proc.returncode:
    print("G1_V3_NEXT=STOP_AND_REVIEW_NO_MODEL_DOWNLOAD")
else:
    print("G1_V3_NEXT=PASS_PREFLIGHT_ONLY")


In [ ]:
# G1 v3: ONE technical smoke, explicitly separate from preflight.
# Only on exact source and same Python/packages/GPU runtime.
import hashlib, importlib.metadata as md, json, pathlib, platform, subprocess, sys
root=pathlib.Path("/content/ttm_six_g1_v3")
script=root/"g1_qwen06_zero_science_smoke_colab_env_20261109_v3.py"
expected_blob="f394293a7c43d8c726d88a1a9b9f63ab49fe3d4a"
if not script.is_file(): raise RuntimeError("STOP_SCRIPT_MISSING")
bytes_=script.read_bytes()
actual_blob=hashlib.sha1(b"blob "+str(len(bytes_)).encode("ascii")+b"\0"+bytes_).hexdigest()
if actual_blob!=expected_blob: raise RuntimeError("STOP_SCRIPT_CHANGED")
receipt_path=root/"G1_PREFLIGHT_RECEIPT.json"
if not receipt_path.is_file(): raise RuntimeError("STOP_PREFLIGHT_RECEIPT_MISSING")
receipt=json.loads(receipt_path.read_text(encoding="utf-8"))
if receipt.get("status")!="PASS_PREFLIGHT_ONLY":
    raise RuntimeError("STOP_PREFLIGHT_NOT_PASS_NO_DOWNLOAD")
if receipt.get("environment_amendment")!="COLAB_RUNTIME_20261109_V3_PY313_PATCH_FLEX":
    raise RuntimeError("STOP_INVALID_ENV_AMENDMENT")
env=receipt.get("environment",{})
if (env.get("python")!=platform.python_version()
    or env.get("python_executable")!=sys.executable
    or env.get("torch")!="2.14.1+cu130"
    or env.get("gpu_name")!="Tesla T4"):
    raise RuntimeError("STOP_RUNTIME_CHANGED_AFTER_PREFLIGHT")
expected_packages={"transformers":"5.22.1","huggingface-hub":"1.33.0",
                   "tokenizers":"0.23.3","safetensors":"0.8.0"}
for name,version in expected_packages.items():
    if md.version(name)!=version: raise RuntimeError("STOP_PACKAGE_CHANGED_"+name)
marker=root/"G1_SMOKE_ATTEMPTED_V3.marker"
with marker.open("x",encoding="utf-8") as stream:
    stream.write("SINGLE_SHOT_CONSUMED_BEFORE_DOWNLOAD\n")
try:
    run=subprocess.run([sys.executable,str(script),"--phase","smoke"],check=False)
    print("G1_V3_SMOKE_EXIT="+str(run.returncode))
    smoke_receipt=root/"G1_SMOKE_RECEIPT.json"
    if smoke_receipt.is_file():
        print("G1_V3_SMOKE_RECEIPT="+smoke_receipt.read_text(encoding="utf-8"))
    else:
        print("G1_V3_SMOKE_RECEIPT=NOT_WRITTEN")
    print("G1_V3_FINAL="+("PASS_TECHNICAL_ONLY" if run.returncode==0 else "STOP_REVIEW_RECEIPT"))
finally:
    print("G1_V3_ONE_SHOT_MARKER=CONSUMED_DO_NOT_REPEAT")
